# Audio — ASR (ubah suara menjadi teks) dengan Whisper
**Kapan dipakai:** soal transkripsi: tiap rekaman ucapan harus ditulis ulang sebagai teks.
Tanpa melatih: model Whisper lokal dipakai apa adanya.

**Langkah:**
1. Muat Whisper-tiny lokal, coba pada beberapa klip.
2. Transkripsi semua klip test, rapikan teks (huruf kecil, buang tanda baca).
3. Simpan submission. Evaluasi WER/CER di sel terakhir (hanya data demo).

**Data demo:** Speech Commands mini, tiap klip satu kata (label asli ada di `test_labels.csv`).

In [1]:
import re

import jiwer
import librosa
import numpy as np
import pandas as pd
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

## Konfigurasi

In [2]:
DATA_FOLDER = "../../../data_latihan/audio/classification"   # GANTI saat lomba
DATA_FOLDER = f"{DATA_FOLDER}/speech_commands_mini"
TEST_CSV = f"{DATA_FOLDER}/test.csv"
TEST_KEY_CSV = f"{DATA_FOLDER}/test_labels.csv"    # kunci jawaban, hanya ada di data demo
WHISPER_FOLDER = "../../../pretrained/whisper_tiny"

ID_COL = "id"
PATH_COL = "path"
SAMPLE_RATE = 16000         # Whisper dilatih dengan 16000 sampel per detik
CLIP_LENGTH = 16000         # 1 detik per klip
LANGUAGE = "en"             # "id" untuk bahasa Indonesia
BATCH_SIZE = 16
MAX_TOKENS = 10             # panjang teks maksimal per klip (naikkan untuk kalimat panjang)
KEEP_PATTERN = r"[^a-z ]"   # buang semua karakter selain huruf kecil dan spasi
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"

## Baca data

In [3]:
test = pd.read_csv(TEST_CSV)
print(test.shape)
test.head()

(320, 3)


,id,path,speaker
0,1,test/down/9e6bb505_nohash_1.wav,9e6bb505
1,2,test/down/dabf67d9_nohash_0.wav,dabf67d9
2,3,test/down/10ace7eb_nohash_3.wav,10ace7eb
3,4,test/down/745a8e32_nohash_0.wav,745a8e32
4,5,test/down/840eab5a_nohash_1.wav,840eab5a


## Langkah 1 — Whisper lokal
`processor` mengubah suara menjadi input Whisper dan token menjadi teks.
`model.generate` menulis teks token demi token.

In [4]:
processor = WhisperProcessor.from_pretrained(WHISPER_FOLDER, local_files_only=True)
model = WhisperForConditionalGeneration.from_pretrained(WHISPER_FOLDER, local_files_only=True)
model.eval()
model = model.to(DEVICE)


def load_clip(path):
    audio, _ = librosa.load(f"{DATA_FOLDER}/{path}", sr=SAMPLE_RATE)
    audio = np.pad(audio, (0, CLIP_LENGTH))     # tambah nol di belakang
    return audio[:CLIP_LENGTH]                  # lalu potong tepat 1 detik


def transcribe(paths):
    texts = []
    for start in range(0, len(paths), BATCH_SIZE):
        clips = []
        for path in paths[start:start + BATCH_SIZE]:
            clips.append(load_clip(path))
        inputs = processor(clips, sampling_rate=SAMPLE_RATE, return_tensors="pt")
        features = inputs.input_features.to(DEVICE)
        with torch.no_grad():
            token_ids = model.generate(features, language=LANGUAGE, task="transcribe",
                                       max_new_tokens=MAX_TOKENS)
        texts += processor.batch_decode(token_ids, skip_special_tokens=True)
    return texts


# coba pada 3 klip
print(transcribe(list(test[PATH_COL][:3])))

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[' Down.', ' down.', ' Yeah.']


## Langkah 2 — transkripsi semua klip, lalu rapikan teks
Whisper menulis "Yes." atau " yes!", padahal jawaban "yes". Teks dibuat huruf kecil dan
tanda baca dibuang supaya bisa dibandingkan.

In [5]:
raw_texts = transcribe(list(test[PATH_COL]))
clean_texts = []
for text in raw_texts:
    text = text.lower()
    text = re.sub(KEEP_PATTERN, "", text)
    clean_texts.append(text.strip())
print(raw_texts[:6])
print(clean_texts[:6])

[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[' Down.', ' down.', ' Yeah.', ' Down.', ' Down.', ' Down.']
['down', 'down', 'yeah', 'down', 'down', 'down']


## Langkah 3 — submission
Tidak ada latih ulang. Kolom `text` berisi transkripsi yang sudah dirapikan.

In [6]:
submission = pd.DataFrame()
submission["id"] = test[ID_COL]
submission["text"] = clean_texts
submission.to_csv("submission.csv", index=False)
submission.head()

,id,text
0,1,down
1,2,down
2,3,yeah
3,4,down
4,5,down


## Cek dengan kunci jawaban (HAPUS saat lomba)
- **WER** (word error rate) = kata yang salah / jumlah kata jawaban. 0 = sempurna.
- **CER** (character error rate) = sama, tapi per huruf.
Tabel di bawah = contoh klip yang salah ditranskripsi.

In [7]:
test_key = pd.read_csv(TEST_KEY_CSV)        # urutan baris sama dengan test.csv
truth = []
for text in test_key["label"]:
    truth.append(text.lower())
print("WER:", round(jiwer.wer(truth, clean_texts), 4))
print("CER:", round(jiwer.cer(truth, clean_texts), 4))

check = pd.DataFrame()
check["truth"] = truth
check["whisper"] = raw_texts
check["clean"] = clean_texts
check.to_csv("transkrip.csv", index=False)
check[check["truth"] != check["clean"]].head(10)

WER: 0.1
CER: 0.0923


,truth,whisper,clean
2,down,Yeah.,yeah
6,down,Now.,now
9,down,the next day.,the next day
13,down,Don't.,dont
29,down,Damn.,damn
36,down,Well...,well
37,down,you,you
66,go,No.,no
70,go,No.,no
91,left,What?,what
